In [1]:
%load_ext autoreload
%autoreload 2
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))
import warnings
warnings.filterwarnings("ignore")

from contra_seq_dataset import ContraSeqDataset
from torch.utils.data import DataLoader, Subset

import random
import numpy as np
random.seed(666)

anc_path = 'model_dataset/anchor_smiles.csv'
aug_path = 'model_dataset/augmented_smiles.csv'

ds_size = 6
ds = ContraSeqDataset(anc_path, aug_path)
print(len(ds))

samp = ds[:10]
# samp = ds.__getitem__(0)
# print(samp)

10000


In [3]:
from seqAE_model import SeqAutoencoder
from contra_seq_dataset import ContraSeqDataset

model = SeqAutoencoder(n_tokens = ds.n_tokens, max_len = 122,
                       dim_emb=512, heads=8, dim_hidden=32,
                       L_enc=6, L_dec=6, dim_ff=2048, 
                       drpt=0.1, actv='relu', eps=0.6, b_first=True)

samp = ds.__getitem__(0)
# print(samp.shape)

print(samp[0])
# anc = samp['anc']
# print(anc)

# from seqAE_model import SeqAutoencoder
# latent_vec, dec_out = model.forward(**anc)
# print(dec_out.shape)
# print(dec_out)

{'smiles': 'Brc1ccc2ccccc2c1', 'atype': 'Anc', 'seq': tensor([11, 22, 27,  5, 27, 27, 27,  6, 27, 27, 27, 27, 27,  6, 27,  5, 13, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24,
        24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24, 24]), 'pad_mask': tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True, 

## Training ?? !!
Pulling this from the `train_step` function in Travis's `training_functions.py`

In [10]:
import torch
from tqdm.auto import trange, tqdm

# use_cuda = False
# device =  torch.device("cuda" if use_cuda else "cpu")

## Model instantiation
lr = 0.00001
optimizer = torch.optim.Adam(model.parameters(), lr = lr)
scheduler = None 
bs = 2
    
model = SeqAutoencoder(n_tokens = ds.n_tokens, max_len = 122,
                       dim_emb=512, heads=8, dim_hidden=32,
                       L_enc=6, L_dec=6, dim_ff=2048, 
                       drpt=0.1, actv='relu', eps=0.6, b_first=True)
## Data loader
# def collate_fn(samp):
#     # custom collate function required bc diff number of augs
#     ancs = [x['anc'] for x in samp]
#     augs = [x['augs'] for x in samp]
#     max_len = max([len(x) for x in augs])
#     for aug_list in augs:
#         while len(aug_list) < max_len:
#             dummy = {k:None for k in ancs[0].keys()}
#             aug_list.append(dummy)
#     return {'anc': ancs,'augs': augs}

def collate_fn(samp):
    # custom collate function required bc diff number of augs
#     print(samp[0])
    max_len = max([len(x) for x in samp])
    for fuzz_ball in samp:
        while len(fuzz_ball) < max_len:
            dummy = {k:None for k in samp[0][0].keys()}
            fuzz_ball.append(dummy)
    return samp

loader = DataLoader(ds, bs, shuffle=True, num_workers=0, pin_memory=True, collate_fn=collate_fn)
data_iter = tqdm(enumerate(loader),total=len(ds))

  0%|          | 0/10000 [00:00<?, ?it/s]

### Check for sizes of stuff in batch ... 

In [13]:
from contra_seq_dataset import ContraSeqDataset
samp = ds[:13]
samp_dum = collate_fn(samp)
print(len(samp_dum[0]))
# print(len(samp_dum['augs'][0]))

11


In [ ]:
for samp in loader:
#     print(samp)
#     print("sdfsfsdfsdfdsfsdfsfffffffff")
    
    anc = samp['anc']
    augs = samp['augs']
    
#     print(len(samp['augs'][0]),"12312312")
    
    labels = np.zeros((n_augs+1)*2)
    labels[:n_augs+1] = 1
#     print(labels)
    
    samps = []
    for fuzz_idx in range(2):
        samps.append(anc[fuzz_idx])
        samps.extend(augs[fuzz_idx])
    print(len(samps))
    print([x['smiles'] for x in samps])
    
    
#     n_augs = len(samp['augs'][0])
#     mask = np.zeros((n_augs+1,n_augs+1))
#     for idx in range(2):
#         print(anc[idx]['smiles'])
#         print(idx)
#     print(samp)

    
#     print(anc[0]['smiles'])
#     print([augs[0][i]['smiles'] for i in range(len(augs[0]))])
    
#     print(len(anc))
#     print(len(augs), len(augs[0]))
    

    break

In [ ]:
    for k in sample.keys():
#         for 
        print(k,v)
        sample[k] = sample[k].to(device)

In [ ]:
train_ds.shape

In [ ]:
# Fitting
for i,samp in data_iter:
    for k,v in samp.items():
        print(k,v)
        sample[key] = sample[key].to(device)

    # Get loss ...
    # 
    #
    #

 

In [ ]:
    loss = train_step(model, sample, optimizer, 
                      kl_alpha = kl_annealer.weight(i),
                     variational = variational,
                     use_out_mask = use_out_mask,
                     prop_pred_loss = prop_pred_loss,
                     recon_alpha = recon_alpha)